# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bytebymanas/fly-rank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

### Rule

Prioritize pages for review when they have meaningful search visibility and show an opportunity for improvement.

I checked two signals:

1. **Content staleness** using `days_since_last_update`
2. **CTR relative to search position** using `ctr` and `avg_position`

The staleness signal was **MIXED**, so I treat it as a secondary signal rather than strong evidence.

The CTR vs position signal was **CONFIRMED**, with median CTR generally decreasing as average position moves from the first page toward position 20+.

### Signal verdicts

- Content staleness: **MIXED**
- CTR vs position: **CONFIRMED**

### Reason codes

- `stale_visible_page` — older content with meaningful search visibility.
- `low_ctr_visible_page` — visible page with weak CTR relative to its search position.
- `stale_and_low_ctr` — page meets both conditions.
- `position_opportunity` — page has meaningful search visibility but ranks below position 20, making it a review candidate.

The score is intended for **review prioritization**, not to guarantee that a page will improve after an action.

In [2]:
import os

print("Current directory:", os.getcwd())
print("\nFiles/folders:")
for item in os.listdir("."):
    print(item)

Current directory: /content

Files/folders:
.config
sample_data


In [3]:
!git clone https://github.com/bytebymanas/fly-rank-ML.git

Cloning into 'fly-rank-ML'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 132 (delta 43), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.88 MiB | 4.78 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [5]:
import pandas as pd

DATA_PATH = "/content/fly-rank-ML/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [8]:
import pandas as pd

import numpy as np

# Use pages with meaningful search visibility for the signal audit
audit = df[df["impressions_90d"] >= 100].copy()

# -----------------------------
# Signal 1: Content staleness
# -----------------------------

audit["staleness_bucket"] = pd.cut(
    audit["days_since_last_update"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90d", "91-180d", "181-365d", "365d+"]
)

staleness_table = (
    audit.groupby("staleness_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_impressions_90d=("impressions_90d", "median"),
        median_ctr=("ctr", "median"),
        median_avg_position=("avg_position", "median")
    )
    .reset_index()
)

print("SIGNAL 1 — CONTENT STALENESS")
print(staleness_table.to_string(index=False))


# -----------------------------
# Signal 2: CTR vs position
# -----------------------------

audit["position_bucket"] = pd.cut(
    audit["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"],
    right=False
)

position_table = (
    audit.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
        median_impressions_90d=("impressions_90d", "median")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR VS SEARCH POSITION")
print(position_table.to_string(index=False))


# -----------------------------
# Simple verdicts
# -----------------------------

age_ctr = staleness_table["median_ctr"].dropna()

if len(age_ctr) >= 3 and age_ctr.iloc[-1] < age_ctr.iloc[0]:
    staleness_verdict = "CONFIRMED"
else:
    staleness_verdict = "MIXED"

pos_ctr = position_table["median_ctr"].dropna()

if len(pos_ctr) >= 3 and pos_ctr.iloc[0] > pos_ctr.iloc[-1]:
    ctr_position_verdict = "CONFIRMED"
else:
    ctr_position_verdict = "MIXED"

print("\nVERDICTS")
print("Content staleness:", staleness_verdict)
print("CTR vs position:", ctr_position_verdict)

SIGNAL 1 — CONTENT STALENESS
staleness_bucket     n  median_impressions_90d  median_ctr  median_avg_position
           0-90d 13887                  1429.0        0.15                11.60
         91-180d  8084                  2286.0        0.13                13.95
        181-365d    35                   429.0        0.18                15.80
           365d+     0                     NaN         NaN                  NaN

SIGNAL 2 — CTR VS SEARCH POSITION
position_bucket    n  median_ctr  mean_ctr  median_impressions_90d
            1-3  503        0.19  0.336382                  2918.0
            3-5 1883        0.32  0.459241                  4997.0
           5-10 6729        0.21  0.325592                  2614.0
          10-20 5932        0.15  0.256472                  1389.0
            20+ 6959        0.04  0.131553                  1030.0

VERDICTS
Content staleness: MIXED
CTR vs position: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# CTR opportunity
# FlyRank's starter baseline treats CTR < 0.5 as the low-CTR threshold.
queue["ctr_opportunity"] = (
    (0.5 - queue["ctr"]).clip(lower=0) / 0.5
)

# Final baseline score
queue["baseline_score"] = 100 * (
    0.40 * queue["visibility_score"] +
    0.20 * queue["staleness_score"] +
    0.25 * queue["position_opportunity"] +
    0.15 * queue["ctr_opportunity"]
)

# Keep pages with meaningful visibility
queue = queue[queue["impressions_90d"] >= 500].copy()


# Exactly ONE reason code per page
def assign_reason(row):
    stale = row["days_since_last_update"] >= 180

    low_ctr = (
        row["impressions_90d"] >= 500
        and 0 < row["avg_position"] <= 20
        and row["ctr"] < 0.5
    )

    position_opportunity = (
        row["impressions_90d"] >= 500
        and row["avg_position"] > 20
    )

    if stale and low_ctr:
        return "stale_and_low_ctr"
    elif stale:
        return "stale_visible_page"
    elif low_ctr:
        return "low_ctr_visible_page"
    elif position_opportunity:
        return "position_opportunity"
    else:
        return "low_ctr_visible_page"


queue["reason_code"] = queue.apply(assign_reason, axis=1)
queue["action"] = "REVIEW"

queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1

output = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "action",
        "reason_code",
        "impressions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update"
    ]
].copy()

import os

output_dir = "/content/fly-rank-ML/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

output.to_csv(output_path, index=False)

print("Queue rows:", len(output))
print("CSV written to:", output_path)

print("\nTop 10:")
display(output.head(10))

Queue rows: 16726
CSV written to: /content/fly-rank-ML/work/outputs/baseline_action_score.csv

Top 10:


,rank,content_id,client_id,baseline_score,action,reason_code,impressions_90d,ctr,avg_position,days_since_last_update
0,1,content_b28d1efd668f,client_6208ef0f77,82.100957,REVIEW,position_opportunity,286608,0.06,26.2,104
1,2,content_813e88069237,client_6208ef0f77,81.478722,REVIEW,position_opportunity,233561,0.06,26.2,104
2,3,content_e752a4e03dd3,client_6208ef0f77,81.218262,REVIEW,position_opportunity,130892,0.01,23.9,104
3,4,content_54baba704595,client_6208ef0f77,81.211868,REVIEW,position_opportunity,130617,0.01,47.0,104
4,5,content_124763d39ca5,client_6208ef0f77,81.192863,REVIEW,position_opportunity,129803,0.01,33.2,104
5,6,content_8b36799b7e44,client_6208ef0f77,81.153022,REVIEW,position_opportunity,141400,0.02,32.0,104
6,7,content_88d367c507a3,client_3fdba35f04,80.319191,REVIEW,position_opportunity,130932,0.04,40.1,104
7,8,content_fb4bf6555c79,client_6208ef0f77,80.173156,REVIEW,position_opportunity,84093,0.00,45.6,104
8,9,content_7368877ea310,client_7f2253d7e2,80.151511,REVIEW,stale_visible_page,59472,0.13,24.8,194
9,10,content_109f8f7c9d39,client_6208ef0f77,80.095576,REVIEW,position_opportunity,90476,0.01,54.4,104


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# Review the top 20 ranked pages

top20 = output.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "low_ctr_visible_page":
        return "High visibility with weak CTR; worth reviewing."
    elif row["reason_code"] == "stale_visible_page":
        return "Older content with meaningful visibility; refresh review may be useful."
    else:
        return "Shows both staleness and CTR opportunity."

def wrong_if(row):
    if row["reason_code"] == "low_ctr_visible_page":
        return "Could be wrong if the low CTR is explained by search intent, SERP features, or an unsuitable query mix."
    elif row["reason_code"] == "stale_visible_page":
        return "Could be wrong if the content is intentionally evergreen or its age is not causing a problem."
    else:
        return "Could be wrong if the page is intentionally old or the CTR reflects the type of search result."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

review = top20[
    [
        "rank",
        "content_id",
        "client_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review.to_string(index=False))

' rank           content_id         client_id action          reason_code                                                         confidence_note                                                                                what_would_make_it_wrong\n    1 content_b28d1efd668f client_6208ef0f77 REVIEW low_ctr_visible_page                         High visibility with weak CTR; worth reviewing. Could be wrong if the low CTR is explained by search intent, SERP features, or an unsuitable query mix.\n    2 content_813e88069237 client_6208ef0f77 REVIEW low_ctr_visible_page                         High visibility with weak CTR; worth reviewing. Could be wrong if the low CTR is explained by search intent, SERP features, or an unsuitable query mix.\n    3 content_e752a4e03dd3 client_6208ef0f77 REVIEW low_ctr_visible_page                         High visibility with weak CTR; worth reviewing. Could be wrong if the low CTR is explained by search intent, SERP features, or an unsuitable query mix.\

## 4. Weak picks + leakage check

Some lower-ranked or borderline pages may be weak picks because CTR can be affected by search intent, SERP features, query mix, or other factors not captured by this simple baseline.

The queue is therefore a **review-prioritization tool**, not a guarantee that changing a page will improve performance.

### Leakage check

The baseline uses only observable fields:

- `impressions_90d`
- `ctr`
- `avg_position`
- `days_since_last_update`

No FlyRank product decision flags such as `health_score`, `priority_score`, `action_type`, `needs_ctr_fix`, or `is_quick_win` were used as features.

No future performance window was used.

In [18]:
# Weak picks + leakage check

print("WEAK PICKS")
print("----------")

# Show the lowest-ranked examples from the reviewed top 20
weak_picks = output.tail(5)[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "impressions_90d",
        "ctr",
        "avg_position"
    ]
]

display(weak_picks)

print("\nLEAKAGE CHECK")
print("-------------")

# Product decision flags must not be used as features
product_flags = [
    "health_score",
    "priority_score",
    "action_type",
    "needs_ctr_fix",
    "is_quick_win"
]

used_columns = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "days_since_last_update"
]

leaked_flags = [col for col in product_flags if col in df.columns]

print("Product flags found in dataset:", leaked_flags)
print("Features used by baseline:", used_columns)

if leaked_flags:
    print("WARNING: Product decision fields are present and must not be used.")
else:
    print("PASS: No product decision flags are present in the dataset.")

print("PASS: No future-window fields were used.")
print("PASS: Score uses observable historical signals only.")

WEAK PICKS
----------


,rank,content_id,baseline_score,reason_code,impressions_90d,ctr,avg_position
16721,16722,content_621a8827feb1,22.598619,low_ctr_visible_page,798,0.50,1.9
16722,16723,content_d92e06dd3ce7,22.486739,low_ctr_visible_page,593,0.67,2.5
16723,16724,content_575564a123ab,22.128660,low_ctr_visible_page,527,0.76,2.5
16724,16725,content_89b7d57d16fd,21.885153,low_ctr_visible_page,688,0.73,1.7
16725,16726,content_40aa55a94b4e,20.740173,low_ctr_visible_page,586,0.51,1.2



LEAKAGE CHECK
-------------
Product flags found in dataset: []
Features used by baseline: ['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update']
PASS: No product decision flags are present in the dataset.
PASS: No future-window fields were used.
PASS: Score uses observable historical signals only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.